<a href="https://colab.research.google.com/github/Roobika-M/deforestation_detector/blob/main/deforest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install segmentation-models-pytorch rasterio flask flask-cors pyngrok earthengine-api -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.7 MB/s eta 0:00:00


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import ee
ee.Authenticate()

# ── Get your project ID ──────────────────────────────────────
# Go to https://console.cloud.google.com
# Look at the top bar — it shows your project name
# Click it → copy the PROJECT ID (looks like "ee-yourname" or "my-project-123")

ee.Initialize(project='amazing-math-417115')
print("GEE Ready ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GEE Ready ✅


In [11]:
import os, threading, time, random
import numpy as np
import torch
import torch.nn as nn
import rasterio
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from pyngrok import ngrok, conf

# ════════════════════════════════════════════════
#  CONFIG — only thing you need to change
# ════════════════════════════════════════════════
NGROK_TOKEN  = "3FMWT68sliPA4ZYjqPf1L6dbhmI_7JRdnL57fyKG8UQePMw5R"
DRIVE_FOLDER = "/content/drive/MyDrive/deforestation_detector"
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
PATCH_SIZE   = 256

# ════════════════════════════════════════════════
#  STEP 1 — Create folders
# ════════════════════════════════════════════════
os.makedirs('/content/app/templates',   exist_ok=True)
os.makedirs('/content/app/static/css',  exist_ok=True)
os.makedirs('/content/app/static/js',   exist_ok=True)
print("Folders ✅")

# ════════════════════════════════════════════════
#  STEP 2 — Train model if checkpoint missing
# ════════════════════════════════════════════════
checkpoint_path = f"{DRIVE_FOLDER}/unet_forest.pth"

if not os.path.exists(checkpoint_path):
    print("\nNo checkpoint found — training now...")

    def load_tif(path):
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)
        return np.nan_to_num(img, nan=0.0)

    def compute_ndvi(img):
        nir, red = img[3].copy(), img[0].copy()
        valid = (nir > 0) | (red > 0)
        ndvi = np.zeros_like(nir)
        ndvi[valid] = (nir[valid]-red[valid])/(nir[valid]+red[valid]+1e-8)
        return ndvi

    before_img = load_tif(f"{DRIVE_FOLDER}/before.tif")
    after_img  = load_tif(f"{DRIVE_FOLDER}/after.tif")
    mask_after = (compute_ndvi(after_img) > 0.4).astype(np.uint8)
    print(f"Images loaded — shape: {before_img.shape}")

    class ForestDataset(Dataset):
        def __init__(self, image, mask, patch=256, n=2000):
            p98 = np.percentile(image[image>0], 98)
            self.img  = np.clip(image/(p98+1e-8), 0, 1).astype(np.float32)
            self.mask = mask
            self.H, self.W = image.shape[1], image.shape[2]
            self.coords = [(random.randint(0,self.H-patch), random.randint(0,self.W-patch)) for _ in range(n)]
            self.patch = patch
        def __len__(self): return len(self.coords)
        def __getitem__(self, i):
            r,c = self.coords[i]; p = self.patch
            im = self.img[:,r:r+p,c:c+p]; mk = self.mask[r:r+p,c:c+p]
            if random.random()>.5: im=im[:,:,::-1].copy(); mk=mk[:,::-1].copy()
            if random.random()>.5: im=im[:,::-1,:].copy(); mk=mk[::-1,:].copy()
            return torch.tensor(im,dtype=torch.float32), torch.tensor(mk,dtype=torch.float32)

    ds = ForestDataset(after_img, mask_after)
    val_n = int(0.1*len(ds))
    train_ds, val_ds = torch.utils.data.random_split(ds, [len(ds)-val_n, val_n])
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2)

    model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet',
                     in_channels=4, classes=1, activation=None).to(DEVICE)

    class DiceBCE(nn.Module):
        def __init__(self): super().__init__(); self.bce=nn.BCEWithLogitsLoss()
        def forward(self,p,t):
            b=self.bce(p,t.unsqueeze(1)); s=torch.sigmoid(p)
            i=(s*t.unsqueeze(1)).sum()
            return b + 1-(2*i+1)/(s.sum()+t.unsqueeze(1).sum()+1)

    criterion = DiceBCE()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
    best = float('inf')

    for epoch in range(20):
        model.train(); tl=0
        for imgs,masks in train_loader:
            imgs,masks=imgs.to(DEVICE),masks.to(DEVICE)
            optimizer.zero_grad(); loss=criterion(model(imgs),masks)
            loss.backward(); optimizer.step(); tl+=loss.item()
        model.eval(); vl=0
        with torch.no_grad():
            for imgs,masks in val_loader:
                imgs,masks=imgs.to(DEVICE),masks.to(DEVICE)
                vl+=criterion(model(imgs),masks).item()
        tl/=len(train_loader); vl/=len(val_loader); scheduler.step()
        if vl<best:
            best=vl; torch.save(model.state_dict(), checkpoint_path); saved="✅"
        else: saved=""
        print(f"Epoch {epoch+1:02d}/20  train={tl:.4f}  val={vl:.4f}  {saved}")

    print(f"Training done — best val: {best:.4f} ✅")
else:
    print(f"Checkpoint found ✅ — skipping training")

# ════════════════════════════════════════════════
#  STEP 3 — Write index.html
# ════════════════════════════════════════════════
with open('/content/app/templates/index.html', 'w') as f:
    f.write('''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>ForestWatch</title>
<link rel="stylesheet" href="/static/css/style.css">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300;400;500;600&family=Space+Mono:wght@400;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.css"/>
</head>
<body>
<div class="grain"></div>
<header>
  <div class="header-inner">
    <div class="logo">
      <div class="logo-icon">
        <svg viewBox="0 0 32 32" fill="none"><circle cx="16" cy="16" r="15" stroke="currentColor" stroke-width="1.5"/>
        <path d="M8 20 Q10 12 16 10 Q22 12 24 20" stroke="currentColor" stroke-width="1.5" fill="none"/>
        <path d="M11 20 Q13 15 16 14 Q19 15 21 20" stroke="currentColor" stroke-width="1.2" fill="none" opacity="0.6"/>
        <line x1="16" y1="20" x2="16" y2="24" stroke="currentColor" stroke-width="1.5"/>
        <line x1="7" y1="24" x2="25" y2="24" stroke="currentColor" stroke-width="1.5" stroke-linecap="round"/></svg>
      </div>
      <span class="logo-text">ForestWatch</span>
    </div>
    <nav>
      <span class="nav-tag">Sentinel-2</span>
      <span class="nav-tag">U-Net</span>
      <div class="status-dot" id="statusDot"></div>
    </nav>
  </div>
</header>
<main>
  <section class="hero">
    <div class="hero-eyebrow">Draw any area · Analyze deforestation</div>
    <h1 class="hero-title">Satellite Forest<br><em>Loss Detection</em></h1>
    <p class="hero-sub">Draw a bounding box anywhere on Earth, pick your date ranges, and the AI detects forest loss using Sentinel-2 imagery and a ResNet-34 U-Net.</p>
  </section>
  <section class="map-section">
    <div class="map-header">
      <span class="map-label">Select area of interest</span>
      <span class="map-hint" id="mapHint">Use the rectangle tool ▭ in the top-left of the map</span>
    </div>
    <div id="map"></div>
    <div class="bbox-display hidden" id="bboxDisplay">
      <span class="bbox-label">Selected</span>
      <span class="bbox-val" id="bboxVal"></span>
      <button class="clear-btn" onclick="clearBbox()">Clear</button>
    </div>
  </section>
  <section class="controls-section">
    <div class="controls-card">
      <div class="controls-grid">
        <div class="control-group">
          <label class="control-label">Before — start</label>
          <input type="date" id="beforeStart" value="2022-01-01">
        </div>
        <div class="control-group">
          <label class="control-label">Before — end</label>
          <input type="date" id="beforeEnd" value="2022-06-30">
        </div>
        <div class="control-group">
          <label class="control-label">After — start</label>
          <input type="date" id="afterStart" value="2023-01-01">
        </div>
        <div class="control-group">
          <label class="control-label">After — end</label>
          <input type="date" id="afterEnd" value="2023-06-30">
        </div>
        <div class="control-group">
          <label class="control-label">Alert threshold</label>
          <div class="slider-row">
            <input type="range" id="threshold" min="0.1" max="10" step="0.1" value="1.0">
            <span class="slider-val" id="thresholdVal">1.0%</span>
          </div>
        </div>
        <div class="control-group">
          <label class="control-label">Sensitivity</label>
          <div class="slider-row">
            <input type="range" id="diffThresh" min="0.01" max="0.2" step="0.01" value="0.05">
            <span class="slider-val" id="diffThreshVal">0.05</span>
          </div>
        </div>
      </div>
      <button class="run-btn" id="runBtn" onclick="runAnalysis()">
        <span class="btn-text">Analyze Selected Area</span>
        <span class="btn-icon"><svg viewBox="0 0 20 20" fill="none" width="18" height="18">
          <circle cx="10" cy="10" r="9" stroke="currentColor" stroke-width="1.5"/>
          <path d="M8 7l5 3-5 3V7z" fill="currentColor"/></svg></span>
      </button>
    </div>
  </section>
  <div id="loadingState" class="loading-state hidden">
    <div class="loading-ring"></div>
    <p class="loading-text" id="loadingText">Fetching satellite imagery…</p>
  </div>
  <section class="results hidden" id="results">
    <div class="alert-banner hidden" id="alertBanner">
      <div class="alert-icon"><svg viewBox="0 0 24 24" fill="none" width="22" height="22">
        <path d="M12 3L2 21h20L12 3z" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round"/>
        <line x1="12" y1="10" x2="12" y2="15" stroke="currentColor" stroke-width="1.5"/>
        <circle cx="12" cy="18" r="0.8" fill="currentColor"/></svg></div>
      <span id="alertText"></span>
    </div>
    <div class="safe-banner hidden" id="safeBanner">
      <div class="alert-icon"><svg viewBox="0 0 24 24" fill="none" width="22" height="22">
        <path d="M12 2L4 6v6c0 5 3.6 9.7 8 11 4.4-1.3 8-6 8-11V6L12 2z" stroke="currentColor" stroke-width="1.5" stroke-linejoin="round"/>
        <path d="M9 12l2 2 4-4" stroke="currentColor" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></svg></div>
      <span id="safeText"></span>
    </div>
    <div class="stats-row">
      <div class="stat-card"><div class="stat-label">Deforestation rate</div><div class="stat-value" id="statPct">—</div></div>
      <div class="stat-card"><div class="stat-label">Forest before</div><div class="stat-value sm" id="statBefore">—</div></div>
      <div class="stat-card"><div class="stat-label">Forest after</div><div class="stat-value sm" id="statAfter">—</div></div>
      <div class="stat-card"><div class="stat-label">Pixels lost</div><div class="stat-value sm" id="statLost">—</div></div>
    </div>
    <div class="imagery-grid">
      <div class="imagery-card">
        <div class="imagery-label"><span class="dot dot-before"></span><span id="labelBefore">Before</span></div>
        <div class="imagery-wrap"><img id="imgBefore" src="" alt="Before"></div>
      </div>
      <div class="imagery-card">
        <div class="imagery-label"><span class="dot dot-after"></span><span id="labelAfter">After</span></div>
        <div class="imagery-wrap"><img id="imgAfter" src="" alt="After"></div>
      </div>
      <div class="imagery-card wide">
        <div class="imagery-label"><span class="dot dot-loss"></span>Forest loss overlay<span class="legend-note">red = detected loss</span></div>
        <div class="imagery-wrap"><img id="imgOverlay" src="" alt="Overlay"></div>
      </div>
    </div>
  </section>
</main>
<footer>
  <span>ForestWatch · Sentinel-2 + ResNet-34 U-Net</span>
  <span id="footerArea">Draw an area to begin</span>
</footer>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script src="https://unpkg.com/leaflet-draw@1.0.4/dist/leaflet.draw.js"></script>
<script src="/static/js/main.js"></script>
</body>
</html>''')
print("index.html ✅")

# ════════════════════════════════════════════════
#  STEP 4 — Write style.css
# ════════════════════════════════════════════════
with open('/content/app/static/css/style.css', 'w') as f:
    f.write('''*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
:root{--moss:#3B5323;--moss-mid:#4A6741;--moss-light:#7A9E6A;--sand-light:#E8D5B0;--cream:#F5EFE0;--bark:#2C1F14;--bark-mid:#3D2C1E;--warning:#C0392B;--warning-bg:#F8E6E4;--safe:#27613A;--safe-bg:#E4F0E8;--text-muted:#6B6459;--text-faint:#9E9589;--border:rgba(60,45,30,0.15);--border-mid:rgba(60,45,30,0.25)}
body{font-family:"Space Grotesk",sans-serif;background:var(--cream);color:var(--bark);min-height:100vh;overflow-x:hidden}
.grain{position:fixed;inset:0;pointer-events:none;opacity:.035;z-index:1000;background-image:url("data:image/svg+xml,%3Csvg viewBox='0 0 256 256' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23n)'/%3E%3C/svg%3E")}
header{position:sticky;top:0;z-index:500;background:rgba(245,239,224,0.92);backdrop-filter:blur(12px);border-bottom:1px solid var(--border)}
.header-inner{max-width:1100px;margin:0 auto;padding:0 2rem;height:60px;display:flex;align-items:center;justify-content:space-between}
.logo{display:flex;align-items:center;gap:10px}
.logo-icon{width:32px;height:32px;color:var(--moss)}
.logo-text{font-size:15px;font-weight:600;letter-spacing:.02em;color:var(--bark)}
nav{display:flex;align-items:center;gap:10px}
.nav-tag{font-family:"Space Mono",monospace;font-size:10px;letter-spacing:.08em;color:var(--moss);background:rgba(59,83,35,.1);border:1px solid rgba(59,83,35,.2);border-radius:4px;padding:3px 8px}
.status-dot{width:8px;height:8px;border-radius:50%;background:#ccc;margin-left:4px;transition:background .3s}
.status-dot.online{background:#3da85e}.status-dot.offline{background:var(--warning)}
main{max-width:1100px;margin:0 auto;padding:0 2rem 4rem}
.hero{padding:5rem 0 3.5rem;border-bottom:1px solid var(--border);margin-bottom:3rem}
.hero-eyebrow{font-family:"Space Mono",monospace;font-size:11px;letter-spacing:.12em;color:var(--moss-light);text-transform:uppercase;margin-bottom:1.2rem}
.hero-title{font-size:clamp(2.5rem,5vw,4rem);font-weight:300;line-height:1.1;letter-spacing:-.02em;color:var(--bark);margin-bottom:1.5rem}
.hero-title em{font-style:normal;font-weight:600;color:var(--moss)}
.hero-sub{max-width:560px;font-size:15px;line-height:1.7;color:var(--text-muted)}
.map-section{margin-bottom:2rem}
.map-header{display:flex;align-items:center;justify-content:space-between;margin-bottom:10px}
.map-label{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.08em;color:var(--text-faint)}
.map-hint{font-family:"Space Mono",monospace;font-size:11px;color:var(--moss-light)}
#map{width:100%;height:420px;border-radius:14px;border:1px solid var(--border);overflow:hidden;z-index:1}
.bbox-display{display:flex;align-items:center;gap:10px;margin-top:10px;padding:10px 14px;background:rgba(59,83,35,.07);border:1px solid rgba(59,83,35,.18);border-radius:8px}
.bbox-label{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.06em;color:var(--moss)}
.bbox-val{font-family:"Space Mono",monospace;font-size:11px;color:var(--bark-mid);flex:1}
.clear-btn{font-size:11px;color:#6B4226;background:none;border:1px solid var(--border-mid);border-radius:5px;padding:4px 10px;cursor:pointer}
.clear-btn:hover{background:var(--sand-light)}
.controls-section{margin-bottom:3rem}
.controls-card{background:#fff;border:1px solid var(--border);border-radius:16px;padding:2rem}
.controls-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:1.5rem;margin-bottom:1.5rem}
@media(max-width:768px){.controls-grid{grid-template-columns:1fr 1fr}}
.control-group{display:flex;flex-direction:column;gap:8px}
.control-label{font-size:11px;font-weight:500;letter-spacing:.08em;text-transform:uppercase;color:var(--text-faint)}
input[type="date"]{font-family:"Space Mono",monospace;font-size:12px;color:var(--bark-mid);background:var(--sand-light);border:1px solid var(--border);border-radius:6px;padding:8px 10px;width:100%;outline:none;cursor:pointer}
input[type="date"]:focus{border-color:var(--moss-light)}
.slider-row{display:flex;align-items:center;gap:10px}
input[type="range"]{flex:1;-webkit-appearance:none;height:3px;background:var(--sand-light);border-radius:2px;outline:none;cursor:pointer}
input[type="range"]::-webkit-slider-thumb{-webkit-appearance:none;width:16px;height:16px;border-radius:50%;background:var(--moss);cursor:pointer;border:2px solid #fff;box-shadow:0 1px 4px rgba(0,0,0,.2)}
.slider-val{font-family:"Space Mono",monospace;font-size:12px;color:var(--bark-mid);min-width:40px;text-align:right}
.run-btn{display:flex;align-items:center;gap:10px;background:var(--moss);color:#fff;border:none;border-radius:10px;padding:14px 28px;font-family:"Space Grotesk",sans-serif;font-size:15px;font-weight:500;cursor:pointer;transition:background .2s;width:100%;justify-content:center}
.run-btn:hover{background:var(--moss-mid)}.run-btn:disabled{background:var(--text-faint);cursor:not-allowed}
.loading-state{display:flex;flex-direction:column;align-items:center;gap:1.2rem;padding:4rem 0}
.loading-ring{width:44px;height:44px;border:3px solid var(--sand-light);border-top-color:var(--moss);border-radius:50%;animation:spin .9s linear infinite}
@keyframes spin{to{transform:rotate(360deg)}}
.loading-text{font-family:"Space Mono",monospace;font-size:13px;color:var(--text-muted)}
.hidden{display:none!important}
.alert-banner,.safe-banner{display:flex;align-items:center;gap:12px;border-radius:10px;padding:14px 18px;margin-bottom:1.5rem;font-size:14px;font-weight:500}
.alert-banner{background:var(--warning-bg);border:1px solid rgba(192,57,43,.25);color:var(--warning)}
.safe-banner{background:var(--safe-bg);border:1px solid rgba(39,97,58,.25);color:var(--safe)}
.alert-icon{flex-shrink:0}
.stats-row{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:2rem}
@media(max-width:640px){.stats-row{grid-template-columns:repeat(2,1fr)}}
.stat-card{background:#fff;border:1px solid var(--border);border-radius:12px;padding:1.2rem 1rem}
.stat-label{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.08em;color:var(--text-faint);margin-bottom:8px}
.stat-value{font-family:"Space Mono",monospace;font-size:26px;font-weight:700;color:var(--bark)}
.stat-value.sm{font-size:16px}
.imagery-grid{display:grid;grid-template-columns:1fr 1fr;gap:16px}
.imagery-card{background:var(--bark);border-radius:14px;overflow:hidden}
.imagery-card.wide{grid-column:1/-1}
.imagery-label{display:flex;align-items:center;gap:8px;padding:12px 16px;font-size:12px;font-weight:500;color:var(--sand-light);border-bottom:1px solid rgba(255,255,255,.08)}
.dot{width:8px;height:8px;border-radius:50%;flex-shrink:0}
.dot-before{background:#7ac47a}.dot-after{background:#e8c87a}.dot-loss{background:#e07a5a}
.legend-note{margin-left:auto;font-size:11px;color:rgba(232,213,176,.5);font-family:"Space Mono",monospace}
.imagery-wrap{width:100%}.imagery-wrap img{width:100%;display:block}
footer{max-width:1100px;margin:0 auto;padding:1.5rem 2rem;border-top:1px solid var(--border);display:flex;justify-content:space-between;font-family:"Space Mono",monospace;font-size:11px;color:var(--text-faint)}''')
print("style.css ✅")

# ════════════════════════════════════════════════
#  STEP 5 — Write main.js
# ════════════════════════════════════════════════
with open('/content/app/static/js/main.js', 'w') as f:
    f.write(r"""
const API_BASE = window.location.origin;
let currentBbox = null;

const map = L.map("map",{zoomControl:true}).setView([0,20],3);
L.tileLayer("https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",{attribution:"© OpenStreetMap",maxZoom:18}).addTo(map);
const drawnItems = new L.FeatureGroup().addTo(map);
map.addControl(new L.Control.Draw({
  draw:{rectangle:{shapeOptions:{color:"#3B5323",weight:2,fillOpacity:.1}},polygon:false,polyline:false,circle:false,marker:false,circlemarker:false},
  edit:{featureGroup:drawnItems,remove:true}
}));
map.on(L.Draw.Event.CREATED,function(e){
  drawnItems.clearLayers(); drawnItems.addLayer(e.layer);
  const b=e.layer.getBounds();
  currentBbox=[parseFloat(b.getWest().toFixed(4)),parseFloat(b.getSouth().toFixed(4)),parseFloat(b.getEast().toFixed(4)),parseFloat(b.getNorth().toFixed(4))];
  document.getElementById("bboxDisplay").classList.remove("hidden");
  document.getElementById("bboxVal").textContent=`W:${currentBbox[0]} S:${currentBbox[1]} E:${currentBbox[2]} N:${currentBbox[3]}`;
  document.getElementById("mapHint").textContent="Area selected — set dates and click Analyze";
  document.getElementById("footerArea").textContent=`${currentBbox[1]}°, ${currentBbox[0]}° → ${currentBbox[3]}°, ${currentBbox[2]}°`;
});
function clearBbox(){
  drawnItems.clearLayers(); currentBbox=null;
  document.getElementById("bboxDisplay").classList.add("hidden");
  document.getElementById("mapHint").textContent="Use the rectangle tool ▭ in the top-left of the map";
  document.getElementById("footerArea").textContent="Draw an area to begin";
}
async function checkHealth(){
  try{
    const d=await(await fetch(`${API_BASE}/api/health`)).json();
    const dot=document.getElementById("statusDot");
    dot.classList.add(d.status==="ok"?"online":"offline");
    dot.title=d.status==="ok"?`Online · ${d.device.toUpperCase()}`:"API offline";
  }catch{document.getElementById("statusDot").classList.add("offline");}
}
const msgs=["Fetching Sentinel-2 imagery from GEE…","Running U-Net inference…","Computing NDVI change signal…","Generating forest loss overlay…","Almost there…"];
async function runAnalysis(){
  if(!currentBbox){alert("Draw a bounding box on the map first.");return;}
  const span=Math.max(currentBbox[2]-currentBbox[0],currentBbox[3]-currentBbox[1]);
  if(span>3){alert("Area too large — draw a box smaller than 3°×3°.");return;}
  const btn=document.getElementById("runBtn"),loading=document.getElementById("loadingState"),results=document.getElementById("results");
  btn.disabled=true; results.classList.add("hidden"); loading.classList.remove("hidden");
  let mi=0; document.getElementById("loadingText").textContent=msgs[0];
  const iv=setInterval(()=>{mi++;document.getElementById("loadingText").textContent=msgs[mi%msgs.length];},7000);
  try{
    const res=await fetch(`${API_BASE}/api/analyze`,{method:"POST",headers:{"Content-Type":"application/json"},
      body:JSON.stringify({bbox:currentBbox,
        before_start:document.getElementById("beforeStart").value,
        before_end:document.getElementById("beforeEnd").value,
        after_start:document.getElementById("afterStart").value,
        after_end:document.getElementById("afterEnd").value,
        threshold:parseFloat(document.getElementById("threshold").value),
        diff_threshold:parseFloat(document.getElementById("diffThresh").value)})});
    const d=await res.json();
    if(!d.success){alert("Error: "+d.error);return;}
    document.getElementById("statPct").textContent=d.deforestation_pct+"%";
    document.getElementById("statBefore").textContent=d.forest_before.toLocaleString();
    document.getElementById("statAfter").textContent=d.forest_after.toLocaleString();
    document.getElementById("statLost").textContent=d.deforested_pixels.toLocaleString();
    document.getElementById("imgBefore").src="data:image/jpeg;base64,"+d.before_img;
    document.getElementById("imgAfter").src="data:image/jpeg;base64,"+d.after_img;
    document.getElementById("imgOverlay").src="data:image/jpeg;base64,"+d.overlay_img;
    document.getElementById("labelBefore").textContent="Before · "+document.getElementById("beforeStart").value+" → "+document.getElementById("beforeEnd").value;
    document.getElementById("labelAfter").textContent="After · "+document.getElementById("afterStart").value+" → "+document.getElementById("afterEnd").value;
    const thr=parseFloat(document.getElementById("threshold").value);
    if(d.alert){
      document.getElementById("alertBanner").classList.remove("hidden");
      document.getElementById("safeBanner").classList.add("hidden");
      document.getElementById("alertText").textContent=`ALERT — ${d.deforestation_pct}% deforestation detected. Exceeds ${thr.toFixed(1)}% threshold.`;
    }else{
      document.getElementById("safeBanner").classList.remove("hidden");
      document.getElementById("alertBanner").classList.add("hidden");
      document.getElementById("safeText").textContent=`Within safe limits — ${d.deforestation_pct}% deforestation detected.`;
    }
    results.classList.remove("hidden");
    results.scrollIntoView({behavior:"smooth",block:"start"});
  }catch(err){alert("Request failed: "+err.message);}
  finally{clearInterval(iv);loading.classList.add("hidden");btn.disabled=false;}
}
document.getElementById("threshold").addEventListener("input",function(){document.getElementById("thresholdVal").textContent=parseFloat(this.value).toFixed(1)+"%";});
document.getElementById("diffThresh").addEventListener("input",function(){document.getElementById("diffThreshVal").textContent=parseFloat(this.value).toFixed(2);});
checkHealth();
""")
print("main.js ✅")

# ════════════════════════════════════════════════
#  STEP 6 — Write app.py + launch Flask + ngrok
# ════════════════════════════════════════════════
with open('/content/app/app.py', 'w') as f:
    f.write(f'''
from flask import Flask, request, jsonify, render_template
from flask_cors import CORS
import numpy as np, torch, ee, requests as req
import segmentation_models_pytorch as smp
import base64, io, os
from PIL import Image

app = Flask(__name__, template_folder="templates", static_folder="static")
CORS(app)
DEVICE = "{"cuda" if torch.cuda.is_available() else "cpu"}"
PATCH_SIZE = 256
DRIVE_FOLDER = "{DRIVE_FOLDER}"

ee.Initialize(project='amazing-math-417115')
_model = None

def get_model():
    global _model
    if _model is None:
        m = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=4, classes=1, activation=None)
        m.load_state_dict(torch.load(f"{{DRIVE_FOLDER}}/unet_forest.pth", map_location=DEVICE))
        _model = m.to(DEVICE); _model.eval()
        print("Model loaded on", DEVICE)
    return _model

def fetch_s2(bbox, start, end, size=512):
    region = ee.Geometry.Rectangle(bbox)
    def mask_clouds(img):
        scl = img.select("SCL")
        return img.updateMask(scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)))
    comp = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(region).filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
            .map(mask_clouds).median().select(["B4","B3","B2","B8"]).clip(region))
    rgb = np.array(Image.open(io.BytesIO(req.get(comp.getThumbURL({{"region":region,"dimensions":size,"format":"png","min":0,"max":3000,"bands":["B4","B3","B2"]}}).replace("\\\\",""), timeout=60).content)).convert("RGB")).astype(np.float32)
    nir = np.array(Image.open(io.BytesIO(req.get(comp.getThumbURL({{"region":region,"dimensions":size,"format":"png","min":0,"max":5000,"bands":["B8","B8","B8"]}}).replace("\\\\",""), timeout=60).content)).convert("L")).astype(np.float32)
    r=rgb[:,:,0]*(3000/255); g=rgb[:,:,1]*(3000/255); b=rgb[:,:,2]*(3000/255); n=nir*(5000/255)
    return np.stack([r,g,b,n], axis=0)

def normalise(img):
    p98 = np.percentile(img[img>0],98) if (img>0).any() else 1.0
    return np.clip(img/(p98+1e-8),0,1).astype(np.float32)

def predict(img_array):
    m=get_model(); img_norm=normalise(img_array)
    _,H,W=img_norm.shape
    pm=np.zeros((H,W),dtype=np.float32); cm=np.zeros((H,W),dtype=np.float32)
    with torch.no_grad():
        for r in range(0,H-PATCH_SIZE+1,PATCH_SIZE//2):
            for c in range(0,W-PATCH_SIZE+1,PATCH_SIZE//2):
                p=torch.tensor(img_norm[:,r:r+PATCH_SIZE,c:c+PATCH_SIZE]).unsqueeze(0).to(DEVICE)
                o=torch.sigmoid(m(p)).squeeze().cpu().numpy()
                pm[r:r+PATCH_SIZE,c:c+PATCH_SIZE]+=o; cm[r:r+PATCH_SIZE,c:c+PATCH_SIZE]+=1
    cm[cm==0]=1; return pm/cm

def compute_ndvi(img):
    nir,red=img[3].copy(),img[0].copy(); valid=(nir>0)|(red>0)
    ndvi=np.zeros_like(nir); ndvi[valid]=(nir[valid]-red[valid])/(nir[valid]+red[valid]+1e-8)
    return ndvi

def make_rgb(arr):
    rgb=np.stack([arr[0],arr[1],arr[2]],axis=-1)
    p2=np.percentile(rgb[rgb>0],2) if (rgb>0).any() else 0
    p98=np.percentile(rgb[rgb>0],98) if (rgb>0).any() else 1
    return np.clip((rgb-p2)/(p98-p2+1e-8),0,1)

def overlay(rgb,mask,alpha=0.55):
    out=rgb.copy(); out[mask==1]=(1-alpha)*rgb[mask==1]+alpha*np.array([0.85,0.25,0.1])
    return out

def to_b64(arr):
    pil=Image.fromarray((np.clip(arr,0,1)*255).astype(np.uint8))
    buf=io.BytesIO(); pil.save(buf,format="JPEG",quality=85)
    return base64.b64encode(buf.getvalue()).decode()

@app.route("/")
def index(): return render_template("index.html")

@app.route("/api/health")
def health(): return jsonify({{"status":"ok","device":DEVICE}})

@app.route("/api/analyze", methods=["POST"])
def analyze():
    try:
        d=request.json or {{}}
        bbox=d.get("bbox",[-52.5,-5.5,-51.5,-4.5])
        bbox[2]=min(bbox[2],bbox[0]+2.0); bbox[3]=min(bbox[3],bbox[1]+2.0)
        bi=fetch_s2(bbox,d.get("before_start","2022-01-01"),d.get("before_end","2022-06-30"))
        ai=fetch_s2(bbox,d.get("after_start","2023-01-01"),d.get("after_end","2023-06-30"))
        pb=predict(bi); pa=predict(ai)
        mb=(pb>0.5).astype(np.uint8); ma=(pa>0.5).astype(np.uint8)
        diff_thresh=float(d.get("diff_threshold",0.05))
        nd=compute_ndvi(bi)-compute_ndvi(ai); pd=pb-pa
        cm=((pd>diff_thresh)|(nd>0.1)).astype(np.uint8)
        total=int(mb.sum()); lost=int(cm.sum())
        pct=round(100*lost/(total+1e-8),2)
        rb=make_rgb(bi); ra=make_rgb(ai); ov=overlay(ra,cm)
        return jsonify({{"success":True,"deforestation_pct":pct,"forest_before":total,
            "forest_after":int(ma.sum()),"deforested_pixels":lost,
            "alert":pct>=float(d.get("threshold",1.0)),
            "before_img":to_b64(rb),"after_img":to_b64(ra),"overlay_img":to_b64(ov)}})
    except Exception as e:
        import traceback
        return jsonify({{"success":False,"error":str(e),"trace":traceback.format_exc()}}),500

if __name__=="__main__":
    get_model()
    app.run(host="0.0.0.0",port=5000,debug=False,use_reloader=False)
''')
print("app.py ✅")

# Kill old processes
os.system("pkill -f 'python /content/app/app.py' 2>/dev/null")
from pyngrok import ngrok
ngrok.kill()
time.sleep(2)

# Launch Flask in background
def run_flask():
    os.system("python /content/app/app.py")

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()
print("\nStarting Flask (loading model)...")
time.sleep(10)

# Check Flask is up
try:
    import requests as req2
    r = req2.get("http://localhost:5000/api/health", timeout=5)
    print("Flask health:", r.json())
except Exception as e:
    print("Flask not responding yet:", e)

# Launch ngrok
conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(5000)
print(f"\n{'='*55}")
print(f"  🌿 ForestWatch is LIVE:")
print(f"  {tunnel.public_url}")
print(f"{'='*55}")
print("\n→ Open the URL, draw a box on the map, set dates, click Analyze")

Folders ✅
Checkpoint found ✅ — skipping training
index.html ✅
style.css ✅
main.js ✅
app.py ✅

Starting Flask (loading model)...
Flask not responding yet: HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x79076e620950>: Failed to establish a new connection: [Errno 111] Connection refused'))

  🌿 ForestWatch is LIVE:
  https://keenly-lapdog-strangle.ngrok-free.dev

→ Open the URL, draw a box on the map, set dates, click Analyze


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!git clone https://github.com/Roobika-M/deforstation_detector.git

Cloning into 'deforstation_detector'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 221 (delta 61), reused 194 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 36.11 MiB | 42.16 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [18]:
!ls /content

app  deforstation_detector  drive  sample_data


In [19]:
%cd /content/deforestation_detector

[Errno 2] No such file or directory: '/content/deforestation_detector'
/content


In [20]:
!git clone https://github.com/Roobika-M/deforestation_detector.git

Cloning into 'deforestation_detector'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 221 (delta 61), reused 194 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 36.11 MiB | 39.51 MiB/s, done.
Resolving deltas: 100% (61/61), done.


In [21]:
%cd /content/deforestation_detector

/content/deforestation_detector


In [24]:
mkdir notebooks

In [26]:
!find /content -name "*.ipynb"

/content/drive/MyDrive/Colab Notebooks/deforest.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled15.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled16.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled17.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled18.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled19.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled20.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled21.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled22.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled23.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled24.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled25.ipynb
/content/drive/MyDrive/Colab Notebooks/transformers.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/ui_in_colab.ipynb
/content/drive/MyDrive/Colab Notebooks/deforestation.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitle

In [27]:
mkdir notebooks

mkdir: cannot create directory ‘notebooks’: File exists


In [28]:
cp "/content/drive/MyDrive/Colab Notebooks/deforest.ipynb" notebooks/